# VAOT datasets: Pouring + Penn Action

Downloads the CARL pre-processed copies (TCC splits and phase labels) from Google Drive, extracts frames, and writes everything to your Drive.

Output in `DRIVE_DIR`:
- `downloads/{pouring,penn_action}/` — original videos and label files (about 0.8 GB, ~2,200 files)
- `vaot_final.tar.gz` — frames + labels in the layout the VAOT code reads (about 4 GB, one file)

Frames are extracted on the Colab local disk and packed into one archive, because ~190k small files are very slow on Drive. IKEA ASM is not included.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/vaot_data'   # change if you like
WORK = '/content/vaot_work'                      # local scratch

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not on Colab: DRIVE_DIR is used as a plain folder')
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)

In [ ]:
# gdown < 6 refuses to list Drive folders with more than 50 files, and Colab ships an older one
!pip -q install -U "gdown>=6.1" opencv-python-headless requests
!python -c "import gdown; print('gdown', gdown.__version__)"

In [ ]:
%%writefile /content/download_vaot_datasets.py
#!/usr/bin/env python3
"""Download the VAOT video-alignment datasets (Pouring, Penn Action).

Source: the pre-processed copies published by CARL (minghchen/CARL_code,
CVPR 2022) on Google Drive. They are the TCC splits and TCC phase labels
(`train.pkl` / `val.pkl` carry per-frame `frame_label`); the original TCC
bucket (storage.googleapis.com/tcc-pouring) returns HTTP 403.

gdown is used only to list the folder; its per-file endpoint gets throttled
after ~60 files, so files are fetched from drive.usercontent.google.com.
Resumable: re-running skips files that already exist.
"""
import argparse
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import gdown
import requests

FOLDERS = {
    "pouring": "1hvA4bDqPnxjiVM4c4mxm-UPOuO1lAVxW",
    "penn_action": "1hPbkKSSM5NoQKKzvAr2bHDp-pCHZpnVj",
}
URL = "https://drive.usercontent.google.com/download?id={}&export=download&confirm=t"


def fetch(item, out, retries=6):
    dst = out / item.path
    if dst.exists() and dst.stat().st_size > 0:
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".part")
    for attempt in range(retries):
        try:
            with requests.get(URL.format(item.id), stream=True, timeout=60) as r:
                ctype = r.headers.get("content-type", "")
                if r.status_code != 200 or "text/html" in ctype:
                    raise RuntimeError(f"status={r.status_code} ctype={ctype}")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            tmp.rename(dst)
            return True
        except Exception as e:  # noqa: BLE001 - retry any transport error
            err = e
            time.sleep(min(120, 5 * 2 ** attempt))
    print(f"FAILED {item.path}: {err}", flush=True)
    return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dest", required=True, type=Path)
    ap.add_argument("--datasets", nargs="+", default=list(FOLDERS), choices=list(FOLDERS))
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args()
    bad = 0
    for name in args.datasets:
        out = args.dest / name
        out.mkdir(parents=True, exist_ok=True)
        items = gdown.download_folder(id=FOLDERS[name], output=str(out), quiet=True, skip_download=True)
        # "._*" entries are macOS resource forks, not data
        items = [i for i in items if not Path(i.path).name.startswith("._")]
        print(f"== {name}: {len(items)} files listed", flush=True)
        with ThreadPoolExecutor(args.threads) as ex:
            ok = list(ex.map(lambda i: fetch(i, out), items))
        bad += ok.count(False)
        print(f"== {name}: {sum(ok)}/{len(items)} present", flush=True)
    sys.exit(1 if bad else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/prepare_vaot_dataset.py
#!/usr/bin/env python3
"""Convert a CARL-format alignment dataset (Pouring / Penn Action) to the
layout the VAOT code reads.

Input  (downloaded by download_vaot_datasets.py):
    <src>/videos/*.mp4, <src>/train.pkl, <src>/val.pkl
    each pkl: list of dicts {name, video_file, frame_label (tensor), seq_len, ...}

Output:
    <dst>/<action>/{train,val}/<video_name>/<idx:04d>.jpg
    <dst>/npyrecords/<action>_{train,val}.npy   {video_name: {'labels': int array}}
    <dst>/manifest.json

Pouring has a single action ("pouring"); Penn Action is split by the
dataset's own action label so one encoder can be trained per action, as the
paper does. Idempotent: a video whose frame count already matches is skipped.
"""
import argparse
import collections
import io
import json
import pickle
from pathlib import Path

import cv2
import numpy as np
import torch

# The pkl files come from a public Google Drive: only allow the constructors
# they legitimately need instead of running arbitrary pickle globals.
_ALLOWED = {
    ("collections", "OrderedDict"): collections.OrderedDict,
    ("torch._utils", "_rebuild_tensor_v2"): torch._utils._rebuild_tensor_v2,
    ("torch.storage", "_load_from_bytes"): lambda b: torch.load(io.BytesIO(b), weights_only=True),
}


ALIASES = {"bowl": "bowling", "jumping_jack": "jumping_jacks", "pushup": "pushups",
           "pullup": "pullups", "squat": "squats", "clean_jerk": "clean_and_jerk"}


class _Restricted(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return _ALLOWED[(module, name)]
        except KeyError:
            raise pickle.UnpicklingError(f"blocked global {module}.{name}")


def load_pkl(path):
    with open(path, "rb") as f:
        return _Restricted(f).load()


def extract(video, out_dir, expect):
    have = len(list(out_dir.glob("*.jpg"))) if out_dir.exists() else 0
    if have == expect:
        return have
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    n = 0
    while True:
        ok, img = cap.read()
        if not ok:
            break
        cv2.imwrite(str(out_dir / f"{n:04d}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        n += 1
    cap.release()
    return n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, type=Path)
    ap.add_argument("--dst", required=True, type=Path)
    ap.add_argument("--single-action", default=None,
                    help="name to use when the dataset has one action (pouring)")
    args = ap.parse_args()

    manifest = {}
    records = collections.defaultdict(dict)
    for split in ("train", "val"):
        items = load_pkl(args.src / f"{split}.pkl")
        if isinstance(items, tuple):  # Penn Action: (videos, per-action index)
            items = items[0]
        for item in items:
            name = item["name"]
            labels = np.asarray(item["frame_label"]).astype(int).reshape(-1)
            # Penn Action names are "<id>_<action>"; use upstream evals/datasets.py spelling
            raw = name.split("_", 1)[1] if not args.single_action else None
            action = args.single_action or ALIASES.get(raw, raw)
            n = extract(args.src / item["video_file"], args.dst / action / split / name, len(labels))
            m = manifest.setdefault(action, {}).setdefault(split, {"videos": 0, "frames": 0, "mismatch": []})
            m["videos"] += 1
            m["frames"] += n
            if n != len(labels):
                m["mismatch"].append([name, n, len(labels)])
            records[(action, split)][name] = {"labels": labels}
    (args.dst / "npyrecords").mkdir(parents=True, exist_ok=True)
    for (action, split), rec in records.items():
        np.save(args.dst / "npyrecords" / f"{action}_{split}.npy", rec, allow_pickle=True)
        manifest[action][split]["label_values"] = sorted(
            int(v) for v in np.unique(np.concatenate([r["labels"] for r in rec.values()])))
    (args.dst / "manifest.json").write_text(json.dumps(manifest, indent=1))
    for action, m in manifest.items():
        for split, s in m.items():
            print(action, split, s["videos"], "videos", s["frames"], "frames",
                  "labels", s["label_values"], "mismatch", len(s["mismatch"]))


if __name__ == "__main__":
    main()


## 1. Download videos and labels to Drive
Resumable: re-run the cell if it stops; finished files are skipped.

In [ ]:
!python /content/download_vaot_datasets.py --dest "$DRIVE_DIR/downloads" --threads 4
!du -sh "$DRIVE_DIR"/downloads/*

## 2. Extract frames and labels (local disk)
Expected: Pouring 70 train / 14 val videos; Penn Action 13 actions, 2,106 videos; `mismatch 0` on every line.

In [ ]:
!python -I /content/prepare_vaot_dataset.py --src "$DRIVE_DIR/downloads/pouring" --dst "$WORK/vaot/pouring" --single-action pouring
!python -I /content/prepare_vaot_dataset.py --src "$DRIVE_DIR/downloads/penn_action" --dst "$WORK/vaot/penn_action"

## 3. Pack into one archive on Drive

In [ ]:
!tar -C "$WORK" -cf - vaot | gzip -1 > "$DRIVE_DIR/vaot_final.tar.gz"
!gzip -t "$DRIVE_DIR/vaot_final.tar.gz" && ls -lh "$DRIVE_DIR/vaot_final.tar.gz"

## Using it later
```bash
mkdir -p reproductions/tssl/data/final
tar -xzf vaot_final.tar.gz -C reproductions/tssl/data/final
```
gives `reproductions/tssl/data/final/vaot/{pouring,penn_action}/...`, the paths `run_vaot.sh` expects.